In [8]:
from dotenv import load_dotenv
!pip install -r requirements.txt

  Obtaining dependency information for llama-index from https://files.pythonhosted.org/packages/79/65/ea61cd261b5b2b2af9130cf9d4a132e8f795ff48eb1eccf4ea21a09dd393/llama_index-0.12.30-py3-none-any.whl.metadata
  Obtaining dependency information for pypdf from https://files.pythonhosted.org/packages/0b/27/d83f8f2a03ca5408dc2cc84b49c0bf3fbf059398a6a2ea7c10acfe28859f/pypdf-5.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for langchain_groq from https://files.pythonhosted.org/packages/c0/58/0d5a19168119c1bd7758ab28d9e6b5c12ba1091bb283f4dc13ca5df7651b/langchain_groq-0.3.2-py3-none-any.whl.metadata
  Obtaining dependency information for groq from https://files.pythonhosted.org/packages/70/77/e6b60636f648922cc53144c36e1e7b77c9670539aab9e8b84a5fd1a53880/groq-0.22.0-py3-none-any.whl.metadata
  Obtaining dependency information for streamlit from https://files.pythonhosted.org/packages/eb/17/fc425e1d4d86e31b2aaf0812a2ef2163763a0670d671720c7c36e8679323/streamlit-1.44.1-py3-none-an

In [9]:
!pip install langchain-community jq

  Obtaining dependency information for jq from https://files.pythonhosted.org/packages/98/0c/2f25f858ee425cfdc9bcc9ec2ec771280d7f807caa81520d106e69f7b7a5/jq-1.8.0-cp39-cp39-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.3/422.3 kB 2.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


## importling libraries

In [1]:
import json
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant
import os

/Users/phoenix/Pycharm Projects/AI-Engine-Prototype/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## helper functions

In [13]:
def get_content_from_record(record):
    # Combine the name, description, and reviews into a single content string
    name = record.get("name", "")
    description = record.get("description", "")
    
    # Format the reviews
    reviews = record.get("reviews", [])
    reviews_text = ""
    if reviews:
        reviews_text = "Reviews:\n" + "\n".join([f"- {review}" for review in reviews])
    
    # Combine everything into content
    content = f"{name}\n\n{description}"
    if reviews_text:
        content += f"\n\n{reviews_text}"
    
    return content

In [14]:
def metadata_func(record, metadata):
    # Extract all fields as metadata
    metadata["id"] = record.get("id", "")
    metadata["name"] = record.get("name", "")
    metadata["tags"] = ", ".join(record.get("tags", []))
    metadata["ambience"] = ", ".join(record.get("ambience", []))
    metadata["popularity"] = record.get("popularity", "")
    metadata["rating"] = record.get("rating", 0)
    metadata["review_sentiment"] = record.get("review_sentiment", "")
    
    opening_hours = record.get("opening_hours", [])
    if len(opening_hours) >= 2:
        metadata["opening_hours"] = f"{opening_hours[0]} - {opening_hours[1]}"
    else:
        metadata["opening_hours"] = ", ".join(opening_hours)
    
    metadata["lat"] = record.get("lat", 0)
    metadata["lon"] = record.get("lon", 0)
    
    busy_hours = record.get("busy_hours", [])
    metadata["busy_hours"] = ", ".join(str(hour) for hour in busy_hours)
    metadata["review_count"] = len(record.get("reviews", []))
    
    return metadata


In [3]:
def content_func(record):
    # Combine the name, description, and reviews into the content
    content_parts = [record.get("name", ""), record.get("description", "")]

    # Add reviews to the content
    reviews = record.get("reviews", [])
    if reviews:
        content_parts.append("Reviews:")
        for review in reviews:
            content_parts.append(f"- {review}")

    return "\n\n".join(content_parts)

In [15]:
def load_json_data(json_path):
    print(f"Loading data from {json_path}")
    
    # Load the JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Process each record manually
    from langchain.schema import Document
    documents = []
    
    for record in data:
        content = get_content_from_record(record)
        metadata = {}
        metadata = metadata_func(record, metadata)
        
        doc = Document(page_content=content, metadata=metadata)
        documents.append(doc)
    
    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    splits = text_splitter.split_documents(documents)
    print(f"Split into {len(splits)} documents")
    return splits

In [7]:
def initialize_embeddings():
    print("Initializing embedding model (baai/bge-m3)")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embedidngs": True}
    )
    return embeddings

In [21]:
def create_vector_store(documents, embeddings):
    persist_directory = "qdrant_db"
    os.makedirs(persist_directory, exist_ok=True)
    
    print(f"create qdrant vector store and persisting to {persist_directory}")
    vector_store = Qdrant.from_documents(
        documents=documents, 
        embedding=embeddings,
        path=persist_directory,
        collection_name="places"
    )
    print(f"Vector store created and persisted successfullly")
    return vector_store

In [9]:
def test_queries(vector_store):
    print("\nTesting queries...")
    
    queries = [
        "Where can I find vegan food?",
        "I'm looking for a cozy place with good reviews",
        "Recommend a popular restaurant"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        results = vector_store.similarity_search(query, k=1)
        
        for i, doc in enumerate(results):
            print(f"Result {i + 1}: {doc.metadata.get('name', 'Unknown')}")
            print(f"Content: {doc.page_content[:100]}...")  # Show first 100 chars
            print(f"Tags: {doc.metadata.get('tags', '')}")
            print(f"Rating: {doc.metadata.get('rating', '')}")
            print(f"Popularity: {doc.metadata.get('popularity', '')}")

In [22]:
def load_existing_vector_store(embeddings):
    persist_directory = "qdrant_db"
    if not os.path.exists(persist_directory):
        print(f"Error: directory {persist_directory} does not exist.")
        return None
    
    print(f"Loading existing Qdrant vector store from {persist_directory}")
    vector_store = Qdrant.from_documents(
        persist_directory=persist_directory,
        embedding=embeddings,
        collection_name="places"
    )
    print("Vector store loaded successfully")
    return vector_store

In [23]:
json_path = "realistic_places_with_location.json"
documents = load_json_data(json_path)

Loading data from realistic_places_with_location.json
Loaded 200 documents


In [24]:
splits = split_documents(documents)

Split into 200 documents


In [25]:
embeddings = initialize_embeddings()
vector_store = create_vector_store(splits, embeddings)

Initializing embedding model (baai/bge-m3)
create qdrant vector store and persisting to qdrant_db
Vector store created and persisted successfullly


In [26]:
test_queries(vector_store)


Testing queries...

Query: Where can I find vegan food?
Result 1: Ryan, Campos and Williams Cafe
Content: Ryan, Campos and Williams Cafe

Free already thought and final candidate reach. Choice thing cause s...
Tags: outdoor seating, romantic, pet friendly, breakfast, quiet
Rating: 4.0
Popularity: moderate

Query: I'm looking for a cozy place with good reviews
Result 1: Schmidt, Robinson and Perkins Cafe
Content: Schmidt, Robinson and Perkins Cafe

Myself pay today huge eye recognize bring. Generation impact dee...
Tags: fast service, pet friendly, luxury, bbq, co-working
Rating: 3.9
Popularity: low

Query: Recommend a popular restaurant
Result 1: Butler and Sons Cafe
Content: Butler and Sons Cafe

Wait political Democrat several. Worry stage ago myself decide whom station. T...
Tags: dinner, casual, lunch, romantic, coffee
Rating: 4.5
Popularity: high


## test QA

In [32]:
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
# Add these imports to your existing code
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import os

# Function to set up the Groq LLM
def setup_groq_llm():
    # Make sure to set your Groq API key as an environment variable
    # You can add this line before running if not already set:
    # os.environ["GROQ_API_KEY"] = "your-groq-api-key"
    
    if "GROQ_API_KEY" not in os.environ:
        raise ValueError("GROQ_API_KEY environment variable is not set. Please set it before running this code.")
    
    print("Setting up Groq LLM...")
    
    # Initialize the Groq LLM
    # You can choose different models like:
    # - llama3-8b-8192
    # - llama3-70b-8192
    # - mixtral-8x7b-32768
    llm = ChatGroq(
        model="llama3-8b-8192",  # You can change this to your preferred model
        temperature=0.2,  # Lower for more factual responses
        max_tokens=1024   # Adjust as needed
    )
    
    return llm

In [37]:
# Function to create a custom prompt template
def create_qa_prompt_template():
    template = """
You are a helpful assistant that helps users find information about places based on their preferences.
You have access to a vector database containing details about various places including restaurants, cafes, and other venues.

Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say you don't know. Don't try to make up an answer.

Context:
{context}

Question: {question}

Provide a detailed and helpful answer:
"""
    return PromptTemplate(
        input_variables=["context", "question"],
        template=template
    )

# Function to create the QA chain
def create_qa_chain(vector_store):
    llm = setup_groq_llm()
    prompt = create_qa_prompt_template()
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  # "stuff" simply stuffs all retrieved documents into the prompt
        retriever=vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}  # Retrieve top 3 most relevant documents
        ),
        return_source_documents=True,  # Return the source documents for reference
        chain_type_kwargs={"prompt": prompt}
    )
    
    return qa_chain

# Function to answer user questions
def answer_question(qa_chain, question):
    print(f"\nQuestion: {question}")
    print("Thinking...")
    
    result = qa_chain.invoke({"query": question})
    
    print("\nAnswer:")
    print(result["result"])
    
    print("\nBased on:")
    for i, doc in enumerate(result["source_documents"]):
        print(f"Source {i+1}: {doc.metadata.get('name', 'Unknown')}")
        print(f"Rating: {doc.metadata.get('rating', 'N/A')}")
        print(f"Tags: {doc.metadata.get('tags', 'N/A')}")
        print("-" * 40)
    
    return result


In [38]:
qa_chain = create_qa_chain(vector_store)
questions = [
    "Can you recommend a vegan restaurant?",
    "Where can i find a cozy place for breakfast?",
    "What's the highest rated restaurant with outdoor seating?"
]

for question in questions:
    answer_question(qa_chain, question)
    print("\n "+ "=" * 50 + "\n")

Setting up Groq LLM...

Question: Can you recommend a vegan restaurant?
Thinking...

Answer:
I'd be happy to help you find a vegan restaurant!

After analyzing the reviews and context provided, I didn't find any specific mention of vegan options or dietary restrictions in the reviews. However, I can try to help you narrow down the options based on the tone and language used in the reviews.

From what I can gather, the reviews seem to focus on the atmosphere, service, and overall experience of the cafes rather than specific menu items or dietary options.

That being said, I don't have enough information to recommend a specific vegan restaurant. If you're looking for vegan options, I would suggest searching online for cafes in the area that specifically mention vegan options or have a separate vegan menu.

If you'd like, I can try to help you find cafes in the area that might have vegan options or are known for being vegan-friendly. Alternatively, you can also try searching online for ve

In [36]:
def main_with_qa():
    # First load your vector database
    embeddings = initialize_embeddings()
    vector_store = load_existing_vector_store(embeddings)
    
    if not vector_store:
        print("No existing vector store found. Creating a new one...")
        json_path = "realistic_places_with_location.json"
        documents = load_json_data(json_path)
        splits = split_documents(documents)
        vector_store = create_vector_store(splits, embeddings)
    
    # Create the QA chain
    qa_chain = create_qa_chain(vector_store)
    
    # Example questions to answer
    questions = [
        "Can you recommend a vegan restaurant?",
        "Where can I find a cozy place for breakfast?",
        "What's the highest rated restaurant with outdoor seating?"
    ]
    
    # Answer each question
    for question in questions:
        answer_question(qa_chain, question)
        print("\n" + "=" * 50 + "\n")
    
    # Interactive mode
    print("\nEnter your questions about places (type 'exit' to quit):")
    while True:
        user_question = input("> ")
        if user_question.lower() in ['exit', 'quit', 'q']:
            break
        answer_question(qa_chain, user_question)
        print("\n" + "=" * 50 + "\n")
